<a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/wl-fraud-detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-dark.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/wl-classification.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Welcome to the WeightsLab image-classification notebook! <a href="https://github.com/GrayboxTech/weightslab">WeightsLab</a> is an open-source PyTorch tool for dataset debugging, mislabel detection, and mid-training data curation. Browse the <a href="https://grayboxtech.github.io/weightslab/latest/index.html">Docs</a> for details, and raise an issue on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a> for support.</div>

# Real Bank Fraud Detection with WeightsLab (tabular)

This notebook trains an MLP on the real Kaggle **[Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)** dataset (ULB): 284,807 European card transactions, only 492 fraudulent (~0.173% prevalence). Every training signal is traced **back to the exact transaction** producing it.

There are no images: a sample is one transaction (a **row**), the model input **is** the 30-feature vector (`Time`, `V1`..`V28`, `Amount`), and WeightsLab sends that vector to the UI as a `vector` (not a fake image). Each raw feature is a **sortable column** in the List Exploration view.

### What you'll do
1. Install WeightsLab (+ `kagglehub` to fetch the real CSV).
2. Download/cache `creditcard.csv` and prepare it (stratified split, train-only oversampling for the extreme imbalance).
3. Wrap the model, optimizer, dataloaders, loss and metrics.
4. Train while streaming per-transaction loss/prediction to Weights Studio.

*At ~0.17% fraud prevalence, plain accuracy is nearly meaningless — predicting "always legit" already scores ~99.8%. This notebook tracks precision/recall/F1/average-precision for the fraud class instead.*

## Setup

Install WeightsLab from PyPI, plus `kagglehub` to fetch the real dataset. These tabular demos are tiny — the free Colab CPU runtime is plenty (no GPU needed).

> The **tabular input path** (the feature vector is sent to the UI as a `vector`, not a fake image) needs a WeightsLab build with tabular support.

<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?color=5865F2&logo=pypi&logoColor=white" alt="PyPI - Version"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/dm/weightslab?color=5865F2" alt="PyPI - Downloads"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/pyversions/weightslab?color=5865F2&logo=python&logoColor=white" alt="PyPI - Python Version"></a>

In [ ]:
%pip install torch torchvision

In [ ]:
%pip install --pre --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "weightslab==1.3.3.dev6"

In [ ]:
%pip install kagglehub

## Get the data

`creditcard.csv` has no anonymous mirror, so getting it takes a one-time step:

* **Recommended** — a free Kaggle account + API token: create one at Kaggle → *Settings* → *API* → *Create New Token*, then upload the resulting `kaggle.json` when prompted below (or, outside Colab, place it at `~/.kaggle/kaggle.json` / export `KAGGLE_USERNAME` + `KAGGLE_KEY`).
* **Manual** — download `creditcard.csv` from the [dataset page](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) yourself and upload it when prompted below instead.

The next cell tries, in order: a path you set in `CSV_PATH_OVERRIDE`, a cached copy from a previous run, then `kagglehub` (which will prompt for the token above) — falling back to a manual file upload in Colab if neither works.

In [ ]:
import os

CSV_PATH_OVERRIDE = None  # e.g. "/content/creditcard.csv" if you already have it
CACHE_DIR = "/content/fraud_cache" if os.path.isdir("/content") else os.path.expanduser("~/.cache/wl_fraud")
os.makedirs(CACHE_DIR, exist_ok=True)
_CACHED_CSV_PATH = os.path.join(CACHE_DIR, "creditcard.csv")


def _resolve_csv_path():
    for candidate in (CSV_PATH_OVERRIDE, _CACHED_CSV_PATH):
        if candidate and os.path.isfile(candidate):
            return candidate

    try:
        import kagglehub
        dataset_dir = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
        downloaded = os.path.join(dataset_dir, "creditcard.csv")
        if os.path.isfile(downloaded):
            return downloaded
    except Exception as e:
        print(f"kagglehub download unavailable/failed ({e}).")

    try:
        from google.colab import files
        print("Please upload creditcard.csv (downloaded from the Kaggle page above):")
        uploaded = files.upload()
        for name in uploaded:
            if name.endswith(".csv"):
                os.replace(name, _CACHED_CSV_PATH)
                return _CACHED_CSV_PATH
    except ImportError:
        pass

    raise FileNotFoundError(
        "Could not find creditcard.csv. Set CSV_PATH_OVERRIDE, install + "
        "authenticate kagglehub, or download it manually from "
        "https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud"
    )


CSV_PATH = _resolve_csv_path()
print(f"Using dataset: {CSV_PATH}")

## 1. Imports

`weightslab` is imported as `wl`. The two `guard_*_context` managers scope a block as training vs. evaluation so signals are attributed to the right phase. We track **precision/recall/F1/average-precision** instead of accuracy — see the intro for why.

In [ ]:
import tempfile, logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchmetrics.classification import Precision, Recall, F1Score, AveragePrecision
from tqdm.auto import tqdm

import weightslab as wl
from weightslab.components.global_monitoring import (
    guard_training_context,
    guard_testing_context,
)

logging.basicConfig(level=logging.ERROR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load, cache, split & rebalance the real dataset (rows, not images)

`creditcard.csv` is parsed once with pandas (`float32` dtypes) and cached as a compressed `.npz` next to it — every re-run after the first skips CSV parsing entirely. The stratified split keeps the natural ~0.17% fraud rate in **both** train and test; the training split is then rebalanced by duplicating fraud rows (with small jitter, so duplicates aren't bit-identical) up to `OVERSAMPLE_FRAUD_RATIO` — otherwise a random batch of a few hundred rows usually contains **zero** fraud examples. The dataset returns the **standardized feature vector** as model input, and `get_items(...)` exposes the **raw** values as metadata columns.

In [ ]:
FEATURE_NAMES = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]
NUM_FEATURES = len(FEATURE_NAMES)  # 30
TEST_SIZE = 0.2
OVERSAMPLE_FRAUD_RATIO = 0.1   # target train-split fraud ratio; None/0 = natural ~1:578
SEED = 0


def _load_raw(csv_path):
    # Cached as a compressed .npz next to the CSV -- the dominant cost for a
    # 144MB file is pandas parsing, so every re-run after the first skips it.
    cache_path = csv_path + ".npz"
    if os.path.isfile(cache_path):
        with np.load(cache_path) as npz:
            return npz["x_raw"], npz["y"]
    dtype = {name: np.float32 for name in FEATURE_NAMES}
    dtype["Class"] = np.int64
    df = pd.read_csv(csv_path, usecols=FEATURE_NAMES + ["Class"], dtype=dtype)
    x_raw = df[FEATURE_NAMES].to_numpy(dtype=np.float32)
    y = df["Class"].to_numpy(dtype=np.int64)
    np.savez_compressed(cache_path, x_raw=x_raw, y=y)
    return x_raw, y


def _stratified_split(y, test_size, seed):
    rng = np.random.default_rng(seed)
    idx_pos, idx_neg = np.where(y == 1)[0], np.where(y == 0)[0]
    rng.shuffle(idx_pos); rng.shuffle(idx_neg)
    n_test_pos = max(1, round(len(idx_pos) * test_size))
    n_test_neg = max(1, round(len(idx_neg) * test_size))
    test_idx = np.concatenate([idx_pos[:n_test_pos], idx_neg[:n_test_neg]])
    train_idx = np.concatenate([idx_pos[n_test_pos:], idx_neg[n_test_neg:]])
    rng.shuffle(test_idx); rng.shuffle(train_idx)
    return train_idx, test_idx


def _oversample_fraud(x_raw, y, target_ratio, seed):
    """Duplicate (+ jitter) fraud rows up to target_ratio -- TRAIN split only."""
    if not target_ratio:
        return x_raw, y
    rng = np.random.default_rng(seed + 1)
    idx_pos, idx_neg = np.where(y == 1)[0], np.where(y == 0)[0]
    n_pos_target = round(target_ratio * len(idx_neg) / (1 - target_ratio))
    n_extra = max(0, n_pos_target - len(idx_pos))
    if n_extra == 0:
        return x_raw, y
    extra_idx = rng.choice(idx_pos, size=n_extra, replace=True)
    extra_x = x_raw[extra_idx].copy()
    feat_std = x_raw[idx_pos].std(axis=0, keepdims=True)
    feat_std[feat_std == 0] = 1.0
    extra_x += rng.normal(0.0, 0.01, extra_x.shape).astype(np.float32) * feat_std
    x_new = np.concatenate([x_raw, extra_x])
    y_new = np.concatenate([y, np.ones(n_extra, np.int64)])
    perm = rng.permutation(len(x_new))
    return x_new[perm], y_new[perm]


def compute_class_weights(y, cap=20.0):
    """Inverse-frequency [legit, fraud] weights, capped so a handful of fraud
    rows at the natural ~1:578 ratio can't dominate the batch loss."""
    n_pos, n_neg = max(1, int((y == 1).sum())), max(1, int((y == 0).sum()))
    w_pos = n_neg / n_pos
    return [1.0, float(min(w_pos, cap) if cap else w_pos)]


class FraudDataset(Dataset):
    """Yields (feature_vector, id, label); get_items() adds raw feature columns."""

    def __init__(self, x_std, x_raw, y):
        self.features = torch.from_numpy(x_std)   # model input (standardized)
        self.raw = x_raw                            # display values (raw scale)
        self.labels = torch.from_numpy(y)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], idx, int(self.labels[idx])

    def get_items(self, idx, include_metadata=False, include_labels=False, include_images=False):
        x = self.features[idx] if include_images else None
        target = int(self.labels[idx]) if include_labels else None
        meta = ({n: round(float(self.raw[idx][i]), 4) for i, n in enumerate(FEATURE_NAMES)}
                if include_metadata else None)
        return x, idx, target, meta


class FraudMLP(nn.Module):
    """BatchNorm helps here specifically because oversampled (duplicated +
    jittered) training rows shift minibatch statistics more than a naturally
    balanced batch would."""

    def __init__(self, in_features=NUM_FEATURES, hidden=64, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, num_classes))

    def forward(self, x):
        return self.net(x)


def build_datasets(csv_path, test_size=TEST_SIZE, oversample_ratio=OVERSAMPLE_FRAUD_RATIO, seed=SEED):
    x_raw, y = _load_raw(csv_path)
    train_idx, test_idx = _stratified_split(y, test_size, seed)
    x_raw_train, y_train = x_raw[train_idx], y[train_idx]
    x_raw_test, y_test = x_raw[test_idx], y[test_idx]

    # Standardization stats from the pre-oversampling TRAIN split only (no leakage).
    mean = x_raw_train.mean(axis=0, keepdims=True)
    std = x_raw_train.std(axis=0, keepdims=True)
    std[std == 0] = 1.0

    x_raw_train, y_train = _oversample_fraud(x_raw_train, y_train, oversample_ratio, seed)

    x_std_train = ((x_raw_train - mean) / std).astype(np.float32)
    x_std_test = ((x_raw_test - mean) / std).astype(np.float32)
    return (FraudDataset(x_std_train, x_raw_train, y_train),
            FraudDataset(x_std_test, x_raw_test, y_test))

## 3. Configuration

Every tunable lives here in one dict, like a `config.yaml` with comments. Wrapping it with `flag="hyperparameters"` lets Weights Studio read (and live-edit) these values while training. `class_weight_cap` bounds a weight that's **computed from the actual training labels** a few cells down (see `compute_class_weights`), not hardcoded — the real ~1:578 prevalence shifts with `oversample_fraud_ratio`, so a fixed value would go stale.

In [ ]:
config = {
    "experiment_name": "fraud_detection_mlp",
    "device": str(device),
    "root_log_dir": tempfile.mkdtemp(prefix="weightslab_fraud_"),
    "learning_rate": 0.005,
    "training_steps_to_do": 2000,
    "eval_full_to_train_steps_ratio": 100,
    "write_export_ratio": 500,
    "class_weight_cap": 20.0,               # bounds the auto-computed fraud weight
    "dataset": {"seed": SEED, "test_size": TEST_SIZE, "oversample_fraud_ratio": OVERSAMPLE_FRAUD_RATIO},
    "data": {"train_loader": {"batch_size": 256},
             "test_loader": {"batch_size": 512}},
}
wl.watch_or_edit(config, flag="hyperparameters", poll_interval=1.0)

## 4. Wrap the training objects

This is the heart of WeightsLab. Each object passes through `wl.watch_or_edit(...)` with a `flag` describing its role. The tracked dataset's `get_items()` exposes every feature as a **sortable column**; `preload_metadata=True` loads them at init. Four metrics (precision/recall/F1/average-precision) are each watched independently so each logs its own signal.

In [ ]:
cfg = config
train_ds, test_ds = build_datasets(
    CSV_PATH, test_size=cfg['dataset']['test_size'],
    oversample_ratio=cfg['dataset']['oversample_fraud_ratio'], seed=cfg['dataset']['seed'])
print(f"train={len(train_ds)} test={len(test_ds)}")

model = wl.watch_or_edit(FraudMLP().to(device), flag='model', device=device)
optimizer = wl.watch_or_edit(
    optim.Adam(model.parameters(), lr=cfg['learning_rate']), flag='optimizer')

train_loader = wl.watch_or_edit(
    train_ds, flag='data', loader_name='train_loader',
    batch_size=cfg['data']['train_loader']['batch_size'], shuffle=True,
    is_training=True, preload_labels=True, preload_metadata=True)
test_loader = wl.watch_or_edit(
    test_ds, flag='data', loader_name='test_loader',
    batch_size=cfg['data']['test_loader']['batch_size'], shuffle=False,
    is_training=False, preload_labels=True, preload_metadata=True)

# Class weights computed from the actual (post-oversampling) training labels.
cw = torch.tensor(compute_class_weights(train_ds.labels.numpy(), cap=cfg['class_weight_cap']),
                  dtype=torch.float32, device=device)
print(f"Class weights [legit, fraud] = {cw.tolist()}")
train_criterion = wl.watch_or_edit(
    nn.CrossEntropyLoss(weight=cw, reduction='none'),
    flag='loss', signal_name='train-loss-CE', log=True)
test_criterion = wl.watch_or_edit(
    nn.CrossEntropyLoss(weight=cw, reduction='none'),
    flag='loss', signal_name='test-loss-CE', log=True)

metrics = {
    'precision': wl.watch_or_edit(Precision(task='binary').to(device), flag='metric',
                                   signal_name='metric-Precision', log=True),
    'recall': wl.watch_or_edit(Recall(task='binary').to(device), flag='metric',
                                signal_name='metric-Recall', log=True),
    'f1': wl.watch_or_edit(F1Score(task='binary').to(device), flag='metric',
                            signal_name='metric-F1', log=True),
    'ap': wl.watch_or_edit(AveragePrecision(task='binary').to(device), flag='metric',
                            signal_name='metric-AveragePrecision', log=True),
}

## 5. Train and evaluate steps

The `guard_training_context` / `guard_testing_context` blocks tell WeightsLab the phase. `criterion(..., batch_ids=ids, preds=preds)` passes the sample ids so loss is stored **per sample**, and `wl.save_signals(...)` logs custom per-sample signals during eval. At ~0.17% fraud prevalence, accuracy alone would hide whether the model does anything useful, so `test()` reports **precision/recall/F1/average-precision** for the fraud class and resets each metric after every eval pass (so each number describes that pass alone, not a running total).

In [ ]:
def train(loader, model, optimizer, criterion, device):
    with guard_training_context:
        inputs, ids, labels = next(loader)
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        preds = logits.argmax(dim=1, keepdim=True)
        loss = criterion(logits.float(), labels.long(), batch_ids=ids, preds=preds)
        total = loss.mean()
        total.backward()
        optimizer.step()
    return total.detach().cpu().item()


def test(loader, model, criterion, metrics, device, n_batches):
    losses = torch.tensor(0.0, device=device)
    for inputs, ids, labels in loader:
        with guard_testing_context:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            preds = logits.argmax(dim=1, keepdim=True)
            probs = torch.softmax(logits, dim=1)[:, 1]
            losses += criterion(logits, labels, batch_ids=ids, preds=preds).mean()

            preds_flat, labels_flat = preds.view(-1), labels.view(-1)
            metrics['precision'].update(preds_flat, labels_flat)
            metrics['recall'].update(preds_flat, labels_flat)
            metrics['f1'].update(preds_flat, labels_flat)
            metrics['ap'].update(probs, labels_flat)

            correct = (preds_flat == labels_flat).float()
            fraud_caught = ((preds_flat == 1) & (labels_flat == 1)).float()
            wl.save_signals(preds_raw=logits, targets=labels, batch_ids=ids,
                            signals={"test_metric/Accuracy_per_sample": correct,
                                     "test_metric/Fraud_caught_per_sample": fraud_caught},
                            preds=preds)
    loss = (losses / max(1, n_batches)).item()
    results = {name: m.compute().item() * 100 for name, m in metrics.items()}
    for m in metrics.values():
        m.reset()
    return loss, results

## 6. Serve and train

`wl.serve(serving_grpc=True, serving_bore=True)` starts the background gRPC server (non-blocking) and a `bore.pub` tunnel so Weights Studio on your own machine can reach this Colab backend. `wl.start_training(...)` flips the experiment into the *training* state, then we run the loop.

In [ ]:
wl.serve(serving_grpc=True, serving_bore=True)

In [ ]:
wl.start_training(timeout=3)

steps = config['training_steps_to_do']
eval_ratio = config['eval_full_to_train_steps_ratio']
n_test_batches = len(test_loader)

test_loss, test_metrics = None, None
pbar = tqdm(range(steps), desc='Training')
for step in pbar:
    age = model.get_age() if hasattr(model, 'get_age') else step
    train_loss = train(train_loader, model, optimizer, train_criterion, device)
    if age > 0 and age % eval_ratio == 0:
        test_loss, test_metrics = test(test_loader, model, test_criterion, metrics, device, n_test_batches)
    postfix = {'loss': f'{train_loss:.3f}'}
    if test_metrics is not None:
        postfix['P'] = f"{test_metrics['precision']:.1f}%"
        postfix['R'] = f"{test_metrics['recall']:.1f}%"
        postfix['F1'] = f"{test_metrics['f1']:.1f}%"
        postfix['AP'] = f"{test_metrics['ap']:.1f}%"
    pbar.set_postfix(postfix)

wl.write_history()
wl.write_dataframe()
print('Training complete.')

## See it live in Weights Studio

Everything above runs headless. The payoff is **Weights Studio**, where each row is one record and every feature is a **sortable column**.

Studio runs as a local Docker stack, and **Colab has no Docker daemon**, so you run Studio on your own machine and point it at this notebook's backend via the `bore.pub:<port>` endpoint **printed in Section 6**.

**On your machine** (with Docker Desktop):
```bash
pip install weightslab
weightslab start                 # opens http://localhost:5173
weightslab tunnel bore.pub:12345     # in another window, the host:port printed in Section 6
```

Then open **http://localhost:5173** and switch the Data Exploration board to the **List** view.

## Curate in the UI

In Weights Studio, switch the Data Exploration board to the **List** view:

1. **Sort by `train-loss-CE` descending** and generate its histogram to find the hardest transactions — usually the frauds the model still misses.
2. **Lock** the loss sort, then add `Amount` or one of the `V1`..`V28` columns as a secondary sort to see which feature ranges drive the errors.
3. **Right-click a column** to clone it, reset the sort, or generate a histogram — the same actions you use on image datasets.
4. **Discard** mislabeled or leaked rows and keep training — no restart.
5. Watch `metric-Precision` / `metric-Recall` / `metric-AveragePrecision` rather than accuracy — at ~0.17% fraud prevalence, accuracy alone can look great while catching almost no fraud.

---

<div align="center">
Crafted by <a href="https://github.com/GrayboxTech/weightslab">GrayboxTech</a> - if WeightsLab helps you catch a bad label, drop us a star on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a>.
</div>